In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql import Row


spark = (
    SparkSession.builder
        .appName("etl-datalake")
        .master("local[*]")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/04/02 12:44:51 WARN Utils: Your hostname, sarti-mint resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface enp3s0)
26/04/02 12:44:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 12:44:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 12:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# RH_FOLHA
dados_rh = [
    Row(cpf="11111111101", nome="João Silva", endereco="Rua A, 100", data_admissao="2020-01-10", salario=5000, data_registro="2023-01-01"),
    Row(cpf="22222222202", nome="Maria Souza", endereco="Rua B, 200", data_admissao="2019-03-15", salario=7000, data_registro="2023-01-01"),
    Row(cpf="33333333303", nome="Carlos Melo", endereco="Rua C, 300", data_admissao="2018-11-20", salario=6000, data_registro="2023-01-01"),
    Row(cpf="44444444404", nome="Fernanda Lima", endereco="Rua D, 400", data_admissao="2022-02-05", salario=4500, data_registro="2023-01-01"),
    Row(cpf="55555555505", nome="Ricardo Alves", endereco="Rua E, 500", data_admissao="2021-06-18", salario=5200, data_registro="2023-01-01"),
]

# ESOCIAL
dados_esocial = [
    Row(cpf="11111111101", nome="João Silva", endereco="Rua A, 100", data_admissao="2020-01-10", salario=4800, data_registro="2023-03-01"),
    Row(cpf="66666666606", nome="Ana Lima", endereco="Rua F, 600", data_admissao="2021-07-01", salario=4000, data_registro="2023-03-01"),
    Row(cpf="33333333303", nome="Carlos Melo", endereco="Rua C, 305", data_admissao="2018-11-20", salario=6100, data_registro="2023-03-01"),  # mudou endereço
    Row(cpf="44444444404", nome="Fernanda Lima", endereco="Rua D, 400", data_admissao="2022-02-05", salario=4500, data_registro="2023-03-01"),
    Row(cpf="77777777707", nome="Bruno Costa", endereco="Rua G, 700", data_admissao="2020-09-10", salario=3900, data_registro="2023-03-01"),
    Row(cpf="87878787802", nome="Amanda Letícia", endereco="Rua H, 800", data_admissao="2019-12-04", salario=5300, data_registro="2023-03-01"),

]


# GESTAO_PESSOAS
dados_gestao = [
    Row(cpf="11111111101", nome="João Silva", endereco="Rua A, 120", data_admissao="2020-01-10", salario=5100, data_registro="2023-05-01"),  # mudou endereço
    Row(cpf="33333333303", nome="Carlos Melo", endereco="Rua C, 305", data_admissao="2018-11-20", salario=6000, data_registro="2023-05-01"),
    Row(cpf="44444444404", nome="Fernanda Lima", endereco="Rua D, 400", data_admissao="2022-02-05", salario=4700, data_registro="2023-05-01"),
    Row(cpf="88888888808", nome="Juliana Rocha", endereco="Rua H, 800", data_admissao="2019-12-01", salario=5300, data_registro="2023-05-01"),
    Row(cpf="55555555505", nome="Ricardo Alves", endereco="Rua E, 550", data_admissao="2021-06-18", salario=5000, data_registro="2023-05-01"),  # mudou endereço
]

In [3]:
df_rh = spark.createDataFrame(dados_rh)
df_esocial = spark.createDataFrame(dados_esocial)
df_gestao = spark.createDataFrame(dados_gestao)


In [4]:
df_rh.write.mode("overwrite").option("header", True).csv("data/raw/rh_folha_pessoas")
df_esocial.write.mode("overwrite").option("header", True).csv("data/raw/esocial_pessoas")
df_gestao.write.mode("overwrite").option("header", True).csv("data/raw/gestao_pessoas")

In [5]:
df_rh = spark.read \
    .option("header", True) \
    .csv("data/raw/rh_folha_pessoas")

df_esocial = spark.read \
    .option("header", True) \
    .csv("data/raw/esocial_pessoas")

df_gestao = spark.read \
    .option("header", True) \
    .csv("data/raw/gestao_pessoas")

In [6]:
df_rh.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|
+-----------+-------------+----------+-------------+-------+-------------+
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |
+-----------+-------------+----------+-------------+-------+-------------+



In [7]:
df_esocial.show(5, False)

+-----------+--------------+----------+-------------+-------+-------------+
|cpf        |nome          |endereco  |data_admissao|salario|data_registro|
+-----------+--------------+----------+-------------+-------+-------------+
|77777777707|Bruno Costa   |Rua G, 700|2020-09-10   |3900   |2023-03-01   |
|87878787802|Amanda Letícia|Rua H, 800|2019-12-04   |5300   |2023-03-01   |
|66666666606|Ana Lima      |Rua F, 600|2021-07-01   |4000   |2023-03-01   |
|33333333303|Carlos Melo   |Rua C, 305|2018-11-20   |6100   |2023-03-01   |
|44444444404|Fernanda Lima |Rua D, 400|2022-02-05   |4500   |2023-03-01   |
+-----------+--------------+----------+-------------+-------+-------------+
only showing top 5 rows



In [8]:
df_gestao.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|
+-----------+-------------+----------+-------------+-------+-------------+
|88888888808|Juliana Rocha|Rua H, 800|2019-12-01   |5300   |2023-05-01   |
|55555555505|Ricardo Alves|Rua E, 550|2021-06-18   |5000   |2023-05-01   |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4700   |2023-05-01   |
|11111111101|João Silva   |Rua A, 120|2020-01-10   |5100   |2023-05-01   |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6000   |2023-05-01   |
+-----------+-------------+----------+-------------+-------+-------------+

